# Notebook 06: ReLU and why nonlinearity matters

In the last notebook, a neuron made a linear value: multiply inputs by weights, add the pieces, then add a bias.
This notebook asks what happens when we put another linear layer after that.
First, we will see that linear layer plus linear layer can still be rewritten as one linear calculation.
Then we add ReLU, a small rule that creates a bend by shutting off negative hidden values.

By the end, you should be able to explain ReLU in two directions:

- forward pass: which hidden values reach the next layer
- backward pass: which gradients are allowed to flow back


### Why two linear layers can collapse into one

Before ReLU, look at the case with no activation function between layers.
An activation function is an extra rule applied after a layer; in this section, there is no extra rule yet.

The `hidden` value is the intermediate result from the first linear layer.
The `output` value is the final result from the second linear layer, which uses `hidden` as its input.

These two lines are like two tiny `nn.Linear` layers with one input and one output each:

```python
hidden = 3 * x + 1
output = 4 * hidden - 5
```

The second layer does **not** add `hidden` and `output` together.
It uses the already-computed `hidden` value to make `output`.

To collapse the chain, replace `hidden` with the formula that created it:

```text
output = 4 * hidden - 5
output = 4 * (3 * x + 1) - 5
output = 12 * x + 4 - 5
output = 12 * x - 1
```

So for `x = 2`, both paths produce `23`.
The word `collapsed` means that the two-step chain has been rewritten as one direct formula from `x` to `output`.

Important lesson: adding a hidden linear layer does not automatically make the model more powerful.
We need a nonlinearity between linear layers so the whole stack cannot flatten back into one straight-line formula.


In [15]:
x = 2

hidden = 3 * x + 1  # like first Linear layer
output = 4 * hidden - 5  # like second Linear layer

collapsed_output = 12 * x - 1  # the final output from the two-layer linear chain, rewritten as one single linear formula
print(output)
print(collapsed_output)

23
23


### ReLU as a forward gate

ReLU stands for Rectified Linear Unit.
For this course, you can think of it as a gate after a hidden linear value.

The rule is:

```text
relu(x) = max(0.0, x)
```

The `max` function picks the larger of two numbers.
That gives ReLU three simple cases:

- if `x < 0.0`, ReLU returns `0.0`
- if `x == 0.0`, ReLU returns `0.0`
- if `x > 0.0`, ReLU returns `x`

This is why the gate metaphor works: negative values are closed off, and positive values pass through unchanged.
The next code cell tests one negative input, zero, and one positive input so we can see the whole rule.


In [16]:
def relu(x: float) -> float:
    return max(0.0, x)

for value in [-2.0, 0.0, 3.0]:
    print(value, "->", relu(value))

-2.0 -> 0.0
0.0 -> 0.0
3.0 -> 3.0


### ReLU derivative as a backward gate

A derivative tells us the local slope: how much the output changes when the input moves a tiny amount.
When `x > 0.0`, ReLU returns `x`, so the slope is `1.0`.
When `x < 0.0`, ReLU stays flat at `0.0`, so the slope is `0.0`.
At exactly `x == 0.0`, ReLU has a corner; for this course, we use `0.0` as the derivative there.

Backward meaning: `1.0` lets the incoming gradient pass through, and `0.0` stops it.


In [17]:
def relu_derivative(x: float) -> float:
    return 1.0 if x > 0.0 else 0.0

In [18]:
for value in [-2.0, 0.0, 3.0]:
    print(value, "->", relu_derivative(value))

-2.0 -> 0.0
0.0 -> 0.0
3.0 -> 1.0


### How ReLU changes backward flow

During backpropagation, a later layer sends a gradient back to this hidden value.
ReLU multiplies that incoming gradient by its derivative.

If the ReLU gate was open, the derivative is `1.0`, so the gradient passes through.
If the ReLU gate was closed, the derivative is `0.0`, so the gradient becomes `0.0`.

In [19]:
gradient_from_later_layer = 5.0

for hidden_before_relu in [-2.0, 3.0]:
    gradient_to_earlier_layer = (
        gradient_from_later_layer * relu_derivative(hidden_before_relu)
    )
    print(hidden_before_relu, "->", gradient_to_earlier_layer)

-2.0 -> 0.0
3.0 -> 5.0
